# Validation of 1d analytical model

In [ ]:
import sys
from os.path import join
# import os
import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt
# import matplotlib.lines as mlines
mm = 1.0/2.54/10

repo_root_path = join('..','..')

python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

# from scm.scm import ilmenite, ilmenite_orig, Abad_SCR, Abad_SCRd, IlmenitePhase, parseReaction, concentration_from_molar_fraction
from scm.scm import ilmenite, IlmenitePhase, Abad_SCR
from labreactor.labreactor_analytical import gas_yield_Leion, oned_Ca_Vdot_vs_hbar, \
    oned_cA_Vdot_vs_hbar_firstord, oned_cA_Vdot_vs_hbar_nthord, oned_cA_Vdot_vs_hbar_odes

plt.style.use(join(repo_root_path, 'src', 'python', 'clc.mplstyle'))

Validation of H2 reaction, using all three approaches.  
Change some parameters and see if they all still agree

In [ ]:
phase = IlmenitePhase(rhoox=3000, Ro=0.033)
reaction = ilmenite['act']['H2']
ms = 15e-3 # [kg] - mass of oxygen carrier
oxidation = 1e-3 # [-] - oxidation [0..1]
T   = 950 + 273.15                              # [K] - temperature
p   = 101325                                    # [Pa] - pressure
Vdot0 = 450 * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
cA0 = p/8.314/T                                 # [mol/m^3] - fuel concentration at the inlet
hbar = np.linspace(0, 1, 1001)


fig, axs = plt.subplots(figsize=[180*mm, 70*mm], ncols = 2, sharey=True)

cA, Vdot = oned_cA_Vdot_vs_hbar_firstord(phase, reaction, ms, oxidation, T, p, 1, Vdot0, hbar)
axs[0].plot(cA/cA0, hbar, label='1st-ord')
axs[1].plot(Vdot/Vdot0, hbar)

rn1001 = Abad_SCR (rstr="H2 + Fe2O3 -> H2O + 2FeO",  rho_m=13590, r_g=0.5e-6,  bavg=1.19, ks0=0.51, Echr=109200, n=1.000001)
oned_cA_Vdot_vs_hbar_nthord(phase, rn1001, ms, oxidation, T, p, 1, Vdot0, hbar)
axs[0].plot(cA/cA0, hbar, '--', label='nth-ord')
axs[1].plot(Vdot/Vdot0, hbar)

cA, Vdot = oned_cA_Vdot_vs_hbar_odes(phase, reaction, ms, oxidation, T, p, 1, Vdot0, hbar)
axs[0].plot(cA/cA0, hbar, ':', label='ODE')
axs[1].plot(Vdot/Vdot0, hbar)

axs[0].set(
    xlabel=r'$c_A(\overline{h}) / c_A(0)$',
    ylabel=r'$\overline{h} = h / h_\mathrm{bed}$',
    # xlim=[0,1],
    # ylim=[0,1]
)
axs[1].set(
    xlabel='$\dot{V}(\overline{h}) / \dot{V}(0)$',
    # xlim=[0.95,3.05]
)
axs[0].legend()

In [ ]:
fig, ax = plt.subplots(figsize=[90*mm, 60*mm])

gammas = []
# for TdegC in [900,950,1000,1050]:
for rhoox in [2500, 3000, 3500, 4000]:
    TdegC = 950
    # rhoox = 3000
    oxidations = np.linspace(0,1,101)
    phase = IlmenitePhase(rhoox=rhoox, Ro=0.033)
    reaction = ilmenite['act']['CH4']
    ms = 15e-3 # [kg] - mass of oxygen carrier
    T   = TdegC + 273.15                            # [K] - temperature
    p   = 101325                                    # [Pa] - pressure
    Vdot0 = 450 * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
    xA0 = 1.0                                       # [-] - mole fraction of fuel at the inlet
    f = np.vectorize(gas_yield_Leion)
    gas_yield = f(phase, reaction, ms, oxidations, T, p, xA0, Vdot0)
    gammas.append(gas_yield)

    omega = 1 - phase.Ro * (1-oxidations)
    # ax.plot(omega, gas_yield, label = f'{TdegC} °C')
    ax.plot(omega, gas_yield, label = f'{rhoox} kg/m^3')
ax.invert_xaxis()
ax.set(
    xlim = [1,0.96],
    ylim = [0,1.0],
    xlabel = '$\omega, -$',
    ylabel = '$\gamma, -$',
)
ax.legend()